# SEC EDGAR Financial Data Pipeline
Pulls quarterly R\&D, Revenue, and Cost of Revenue from the SEC EDGAR XBRL API
for companies in our cleaned patent database.

**Output files (written to this directory):**
- `sec_filers.csv` – companies matched to a SEC CIK
- `non_sec_filers.csv` – companies with no SEC filing found
- `sec_financials.csv` – quarterly financial data matching the screenshot format

In [ ]:
import requests, json, time, re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# SEC requires a descriptive User-Agent
HEADERS = {
    "User-Agent": "Energy Economics Lab research@energyeconlab.edu",
    "Accept-Encoding": "gzip, deflate"
}
SLEEP = 0.22   # ~4.5 req/s (below SEC 10/s hard limit)
BASE  = Path("/Users/humza/energy_economics_lab_work/Energy_Econ_Lab/Company R&D")
CLEANED_CSV = "/Users/humza/Downloads/companies_cleaned.csv"

In [ ]:
# ── Hardcoded CIKs (verified against EDGAR) ──────────────────────────────────
# Format: "Official Name": (cik_str, primary_form_type)
# None = known non-filer (skip EDGAR lookup)
KNOWN_CIKS = {
    # US domestic 10-K / 10-Q filers
    "Nvidia Corporation":                        ("1045810",  "10-K/10-Q"),
    "Intel Corporation":                         ("50863",    "10-K/10-Q"),
    "Micron Technology, Inc.":                   ("723125",   "10-K/10-Q"),
    "Advanced Micro Devices, Inc.":              ("2488",     "10-K/10-Q"),
    "Applied Materials, Inc.":                   ("3153",     "10-K/10-Q"),
    "Qualcomm Incorporated":                     ("804328",   "10-K/10-Q"),
    "Texas Instruments Incorporated":            ("97476",    "10-K/10-Q"),
    "Alphabet Inc.":                             ("1652044",  "10-K/10-Q"),
    "Microsoft Corporation":                     ("789019",   "10-K/10-Q"),
    "Apple Inc.":                                ("320193",   "10-K/10-Q"),
    "Amazon.com, Inc.":                          ("1018724",  "10-K/10-Q"),
    "Meta Platforms, Inc.":                      ("1326801",  "10-K/10-Q"),
    "Dell Technologies Inc.":                    ("1571996",  "10-K/10-Q"),
    "IBM Corporation":                           ("51143",    "10-K/10-Q"),
    "Western Digital Corporation":               ("106040",   "10-K/10-Q"),
    "Lam Research Corporation":                  ("707549",   "10-K/10-Q"),
    "Cisco Systems, Inc.":                       ("858877",   "10-K/10-Q"),
    "Oracle Corporation":                        ("1341439",  "10-K/10-Q"),
    "Adobe Inc.":                                ("796343",   "10-K/10-Q"),
    "Salesforce, Inc.":                          ("1108524",  "10-K/10-Q"),
    "VMware LLC":                                ("1124610",  "10-K/10-Q"),
    "Snap Inc.":                                 ("1564408",  "10-K/10-Q"),
    "PayPal Holdings, Inc.":                     ("1633917",  "10-K/10-Q"),
    "Visa Inc.":                                 ("1403161",  "10-K/10-Q"),
    "Mastercard Incorporated":                   ("1141391",  "10-K/10-Q"),
    "Wells Fargo & Company":                     ("72971",    "10-K/10-Q"),
    "JPMorgan Chase & Co.":                      ("19617",    "10-K/10-Q"),
    "Bank of America Corporation":               ("70858",    "10-K/10-Q"),
    "Capital One Financial Corporation":         ("927628",   "10-K/10-Q"),
    "Honeywell International Inc.":              ("773840",   "10-K/10-Q"),
    "GE Aerospace":                              ("40987",    "10-K/10-Q"),
    "Carrier Global Corporation":                ("1783398",  "10-K/10-Q"),
    "GlobalFoundries Inc.":                      ("1709626",  "10-K/10-Q"),
    "Wolfspeed, Inc.":                           ("895419",   "10-K/10-Q"),
    "Monolithic Power Systems, Inc.":            ("1280452",  "10-K/10-Q"),
    "onsemi (ON Semiconductor)":                 ("861284",   "10-K/10-Q"),
    "Microchip Technology Incorporated":         ("827054",   "10-K/10-Q"),
    "Synopsys, Inc.":                            ("883241",   "10-K/10-Q"),
    "Cadence Design Systems, Inc.":              ("813672",   "10-K/10-Q"),
    "Silicon Laboratories Inc.":                 ("1038074",  "10-K/10-Q"),
    "MACOM Technology Solutions Holdings, Inc.": ("1493594",  "10-K/10-Q"),
    "ServiceNow, Inc.":                          ("1373715",  "10-K/10-Q"),
    "Pure Storage, Inc.":                        ("1474735",  "10-K/10-Q"),
    "Snowflake Inc.":                            ("1640147",  "10-K/10-Q"),
    "NetApp, Inc.":                              ("1002047",  "10-K/10-Q"),
    "Arista Networks, Inc.":                     ("1313925",  "10-K/10-Q"),
    "Juniper Networks, Inc.":                    ("1043604",  "10-K/10-Q"),
    "Ciena Corporation":                         ("936395",   "10-K/10-Q"),
    "Corning Incorporated":                      ("24741",    "10-K/10-Q"),
    "Equifax Inc.":                              ("33185",    "10-K/10-Q"),
    "Fair Isaac Corporation (FICO)":             ("814547",   "10-K/10-Q"),
    "Palo Alto Networks, Inc.":                  ("1327567",  "10-K/10-Q"),
    "Illumina, Inc.":                            ("1110803",  "10-K/10-Q"),
    "eBay Inc.":                                 ("1013239",  "10-K/10-Q"),
    "Dropbox, Inc.":                             ("1467623",  "10-K/10-Q"),
    "Intuit Inc.":                               ("896878",   "10-K/10-Q"),
    "Tesla, Inc.":                               ("1318605",  "10-K/10-Q"),
    "Seagate Technology Holdings plc":           ("1137789",  "10-K/10-Q"),
    "Kyndryl Holdings, Inc.":                    ("1835079",  "10-K/10-Q"),
    "Allegro MicroSystems, Inc.":                ("1816090",  "10-K/10-Q"),
    "Power Integrations, Inc.":                  ("833640",   "10-K/10-Q"),
    "Navitas Semiconductor Limited":             ("1801170",  "10-K/10-Q"),
    "Analog Devices, Inc.":                      ("6951",     "10-K/10-Q"),
    "Skyworks Solutions, Inc.":                  ("4127",     "10-K/10-Q"),
    "Booz Allen Hamilton Inc.":                  ("1443646",  "10-K/10-Q"),
    "Palantir Technologies Inc.":                ("1321655",  "10-K/10-Q"),
    "Rambus Inc.":                               ("917273",   "10-K/10-Q"),
    "Marvell Technology, Inc.":                  ("1058057",  "10-K/10-Q"),
    "Entegris, Inc.":                            ("1101302",  "10-K/10-Q"),
    "Diodes Incorporated":                       ("29002",    "10-K/10-Q"),
    "Adeia Inc.":                                ("1840292",  "10-K/10-Q"),
    "Accenture plc":                             ("1467373",  "10-K/10-Q"),
    "Broadcom Inc.":                             ("1730168",  "10-K/10-Q"),
    "NXP Semiconductors N.V.":                   ("1413447",  "10-K/10-Q"),
    "Alpha and Omega Semiconductor Limited":     ("1417398",  "10-K/10-Q"),
    "D-Wave Systems Inc.":                       ("1907982",  "10-K/10-Q"),
    # Foreign 20-F filers on US exchanges
    "Taiwan Semiconductor Manufacturing Company Limited": ("1046179", "20-F"),
    "Arm Holdings plc":                          ("1748727",  "20-F"),
    "STMicroelectronics N.V.":                   ("310764",   "20-F"),
    "Ericsson":                                  ("101829",   "20-F"),
    "ASE Technology Holding Co., Ltd.":          ("1492812",  "20-F"),
    "SAP SE":                                    ("935576",   "20-F"),
    "Sony Group Corporation":                    ("313838",   "20-F"),
    "Canon Inc.":                                ("26986",    "20-F"),
    "United Microelectronics Corporation":       ("1094689",  "20-F"),
    # Known non-filers
    "Samsung Electronics Co., Ltd.":            None,
    "SK Hynix Inc.":                            None,
    "Huawei Technologies Co., Ltd.":            None,
    "Kioxia Corporation":                        None,
    "Nanya Technology Corporation":             None,
    "Mitsubishi Electric Corporation":          None,
    "DENSO Corporation":                         None,
    "Murata Manufacturing Co., Ltd.":           None,
    "Tokyo Electron Limited":                    None,
    "Renesas Electronics Corporation":           None,
    "Hitachi, Ltd.":                            None,
    "Fujitsu Limited":                           None,
    "Toshiba Corporation":                       None,
    "Infineon Technologies AG":                  None,
    "Robert Bosch GmbH":                        None,
    "MediaTek Inc.":                             None,
    "Tencent Holdings Limited":                  None,
    "Baidu, Inc.":                               None,
    "Yangtze Memory Technologies Co., Ltd.":    None,
    "Semiconductor Manufacturing International Corporation": None,
    "Changxin Memory Technologies, Inc.":        None,
}

In [ ]:
# ── Load cleaned company list ─────────────────────────────────────────────────
df_co = pd.read_csv(CLEANED_CSV)
print(f"{len(df_co):,} canonical companies loaded")
df_co.head(10)

In [ ]:
# ── Download EDGAR company tickers (one-time, ~1 MB) ─────────────────────────
print("Fetching EDGAR tickers...")
r = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS)
r.raise_for_status()
tickers_raw = r.json()

def _norm(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

edgar_lookup = {}   # normalised_name -> (cik_str, ticker, title)
for entry in tickers_raw.values():
    norm = _norm(entry["title"])
    edgar_lookup[norm] = (str(entry["cik_str"]), entry.get("ticker",""), entry["title"])

print(f"{len(edgar_lookup):,} EDGAR companies indexed")

In [ ]:
# ── Match each company to a CIK ──────────────────────────────────────────────
# Priority: (1) KNOWN_CIKS  (2) exact normalised name in EDGAR  (3) not found

def find_cik(official_name):
    if official_name in KNOWN_CIKS:
        val = KNOWN_CIKS[official_name]
        if val is None:
            return None, None, "known non-filer"
        return val[0], val[1], "hardcoded"
    norm = _norm(official_name)
    if norm in edgar_lookup:
        cik, _, _ = edgar_lookup[norm]
        return cik, "10-K/10-Q", "edgar-exact"
    # Partial: first 25 chars match (handles Inc./LLC suffix variants)
    for key, val in edgar_lookup.items():
        if len(key) > 10 and norm.startswith(key[:25]):
            return val[0], "10-K/10-Q", "edgar-partial"
    return None, None, "not found"

results = []
for _, row in df_co.iterrows():
    cik, ftype, source = find_cik(row["official_name"])
    results.append({
        "official_name":   row["official_name"],
        "entity_type":     row["entity_type"],
        "total_patents":   row["total_patents"],
        "total_citations": row["total_citations"],
        "notes":           row["notes"],
        "raw_variants":    row["raw_name_variants"],
        "cik":             cik,
        "form_type":       ftype,
        "cik_source":      source,
    })

df_matched     = pd.DataFrame(results)
sec_filers     = df_matched[df_matched["cik"].notna()].copy()
non_sec_filers = df_matched[df_matched["cik"].isna()].copy()

print(f"SEC filers found : {len(sec_filers):,}")
print(f"No SEC filing    : {len(non_sec_filers):,}")
sec_filers[["official_name","cik","form_type","cik_source","total_patents"]].head(20)

In [ ]:
# ── Save split CSVs ───────────────────────────────────────────────────────────
sec_filers.to_csv(BASE / "sec_filers.csv", index=False)
non_sec_filers.to_csv(BASE / "non_sec_filers.csv", index=False)
print("Saved sec_filers.csv and non_sec_filers.csv")

In [ ]:
# ── XBRL helpers ─────────────────────────────────────────────────
REVENUE_TAGS = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
    "SalesRevenueNet",
    "SalesRevenueGoodsNet",
    "NetRevenues",
]
COR_TAGS = [
    "CostOfRevenue",
    "CostOfGoodsAndServicesSold",
    "CostOfGoodsSold",
    "CostOfServices",
]
RND_TAGS = [
    "ResearchAndDevelopmentExpense",
    "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
]

def _pad_cik(cik): return str(cik).zfill(10)

def _filing_url(cik, accn):
    return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accn.replace('-','')}/"

def _best_period_fact(facts, td, tol=30):
    """
    Among facts whose period length is within tol days of td,
    return the one with the LATEST end date.
    This selects the current reporting period, not prior-year comparisons
    that companies include in the same filing.
    """
    candidates = []
    for f in facts:
        if "start" not in f:
            continue
        try:
            days = (pd.to_datetime(f["end"]) - pd.to_datetime(f["start"])).days
        except Exception:
            continue
        if abs(days - td) <= tol:
            candidates.append(f)
    if not candidates:
        return None
    return max(candidates, key=lambda f: f["end"])

def extract_metric(usd_facts, forms, target_days):
    """Return {accn: {val, filed, end, form}} — one entry per filing."""
    by_accn = {}
    for f in usd_facts:
        if f.get("form") not in forms:
            continue
        by_accn.setdefault(f["accn"], []).append(f)
    out = {}
    for accn, facts in by_accn.items():
        chosen = _best_period_fact(facts, target_days)
        if chosen:
            out[accn] = {"val": chosen["val"], "filed": chosen.get("filed",""),
                         "end": chosen["end"], "form": chosen["form"]}
    return out

def first_available(gaap, tags, forms, target_days):
    """Try each XBRL tag; return first non-empty extracted dict."""
    for tag in tags:
        facts = gaap.get(tag, {}).get("units", {}).get("USD", [])
        if facts:
            result = extract_metric(facts, forms, target_days)
            if result:
                return result, tag
    return {}, None

def fetch_company_financials(cik, company_name, forms):
    """Fetch XBRL facts for one company. Returns list of row dicts."""
    target_days = 365 if forms == {"20-F"} else 91
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{_pad_cik(cik)}.json"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        time.sleep(SLEEP)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ERROR {company_name}: {e}")
        return []
    gaap = resp.json().get("facts", {}).get("us-gaap", {})
    if not gaap:
        return []
    rnd_data, _ = first_available(gaap, RND_TAGS,     forms, target_days)
    rev_data, _ = first_available(gaap, REVENUE_TAGS, forms, target_days)
    cor_data, _ = first_available(gaap, COR_TAGS,     forms, target_days)
    rows = []
    for accn in set(rnd_data) | set(rev_data) | set(cor_data):
        meta = rnd_data.get(accn) or rev_data.get(accn) or cor_data.get(accn)
        rows.append({
            "Company":         company_name,
            "Form Type":       meta["form"],
            "Filing date":     meta["filed"],
            "Period end":      meta["end"],
            "R&D Expenses":    round(rnd_data[accn]["val"] / 1e6, 2) if accn in rnd_data else None,
            "Revenue":         round(rev_data[accn]["val"] / 1e6, 2) if accn in rev_data else None,
            "Cost of Revenue": round(cor_data[accn]["val"] / 1e6, 2) if accn in cor_data else None,
            "URL":             _filing_url(cik, accn),
        })
    return rows

print("Helpers defined.")

In [ ]:
# ── Smoke test: Nvidia (should match the uploaded screenshot) ─────────────────
test_rows = fetch_company_financials("1045810", "Nvidia Corporation", {"10-Q"})
df_test = pd.DataFrame(test_rows).sort_values("Filing date")
print(f"Nvidia: {len(df_test)} quarterly filings retrieved")
df_test[["Company","Form Type","Filing date","R&D Expenses","Revenue","Cost of Revenue"]].head(10)

In [ ]:
# ── Fetch financials for all SEC filers ───────────────────────────────────────
# 10-Q for domestic filers; 20-F for foreign annual filers.
# Runtime: ~0.22s x N companies.

all_rows, failed = [], []

for _, row in sec_filers.iterrows():
    name  = row["official_name"]
    cik   = row["cik"]
    ftype = str(row["form_type"])
    forms = {"20-F"} if "20-F" in ftype and "10" not in ftype else {"10-Q"}
    print(f"  {name} (CIK {cik}) ...", end=" ")
    rows = fetch_company_financials(cik, name, forms)
    if rows:
        all_rows.extend(rows)
        print(f"{len(rows)} periods")
    else:
        failed.append(name)
        print("no data")

print(f"
Total rows: {len(all_rows):,}")
if failed:
    print("No data:", failed)

In [ ]:
# ── Build final DataFrame ────────────────────────────────────────────────────
df = pd.DataFrame(all_rows)
df["Filing date"] = pd.to_datetime(df["Filing date"])
df = (
    df
    .dropna(subset=["R&D Expenses","Revenue","Cost of Revenue"], how="all")
    .sort_values(["Company", "Filing date"])
    .reset_index(drop=True)
)
print(f"{len(df):,} rows | {df['Company'].nunique()} companies")
print(f"Date range: {df['Filing date'].min().date()} to {df['Filing date'].max().date()}")
df[["Company","Form Type","Filing date","R&D Expenses","Revenue","Cost of Revenue","URL"]].head(30)

In [ ]:
# ── Coverage summary ─────────────────────────────────────────────────────────
summary = (
    df.groupby("Company")
    .agg(
        filings      = ("Filing date",      "count"),
        first_filing = ("Filing date",      "min"),
        last_filing  = ("Filing date",      "max"),
        has_rnd      = ("R&D Expenses",     lambda x: x.notna().sum()),
        has_revenue  = ("Revenue",          lambda x: x.notna().sum()),
        has_cor      = ("Cost of Revenue",  lambda x: x.notna().sum()),
    )
    .reset_index()
    .sort_values("filings", ascending=False)
)
print(summary.to_string(index=False))

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
out = BASE / "sec_financials.csv"
df[["Company","Form Type","Filing date","Period end",
    "R&D Expenses","Revenue","Cost of Revenue","URL"]].to_csv(out, index=False)
print(f"Saved {len(df):,} rows to {out}")